# Notebook 6 (extended) — TinyGPT on *The Wonderful Wizard of Oz*

My own extension of Notebook 06: the same Pre-LN Transformer, trained on the Oz text in
`input.txt` instead of Tiny Shakespeare, with six changes:

- GELU instead of ReLU in the feed-forward network
- weight tying between the token embedding and the LM head
- gradient clipping (`max_norm=1.0`)
- a held-out validation set: the first 80% of the text for training, the last 20% for validation,
  with training windows and validation windows built separately so they never overlap
- a cosine learning-rate schedule
- temperature and top-k sampling


In [1]:
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

torch.manual_seed(1337)

if not Path("input.txt").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/h23yonsei/tinygpt-transformer-from-scratch/main/input.txt", "input.txt")
text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 64

# Hold out the last 20% of the text for validation. The windows are built separately inside each
# part, so no validation window shares characters with a training window.
n = int(0.8 * len(data))
train_dataset = NextTokenDataset(data[:n], block_size)
val_dataset = NextTokenDataset(data[n:], block_size)
loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

xb, yb = next(iter(loader))

## 1. Multi-Head Attention

In [2]:
class Head(nn.Module):
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = emb_dim // num_heads
        self.heads = nn.ModuleList([Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

## 2. Feedforward + Block

In [3]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.GELU(),  # GELU handles negative inputs smoothly, giving better gradient flow than ReLU (standard since GPT-2)
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.sa = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## 3. Tiny GPT

In [4]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size, bias=False)
        # Weight tying: token_embedding and lm_head share weights
        # The input embedding and output projection use the same vector space,
        # which reduces parameters and improves quality (GPT-2 does the same)
        self.lm_head.weight = self.token_embedding.weight

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]
        h = tok + pos
        h = self.blocks(h)
        h = self.ln_f(h)
        logits = self.lm_head(h)
        return logits

model = TinyGPT(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 64, 68])


## 4. Training

In [5]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping: prevents exploding gradients (max_norm=1.0 is the usual value)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

@torch.no_grad()
def eval_loss(model, loader, device):
    # Compute validation loss — inference only, no gradients
    model.eval()
    total_loss, total_count = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyGPT(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
# CosineAnnealingLR: gradually lowers the learning rate late in training to stabilize convergence
# which converges to a better final loss than a fixed lr
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    val = eval_loss(model, val_loader, device)
    scheduler.step()
    print(f"epoch {epoch:2d} | train {train_loss:.4f} | val {val:.4f}")

epoch  0 | train 6.4474 | val 2.4425
epoch  1 | train 2.5043 | val 2.2678
epoch  2 | train 2.2510 | val 2.0843
epoch  3 | train 2.1010 | val 1.9779
epoch  4 | train 1.9943 | val 1.8957
epoch  5 | train 1.8993 | val 1.8105
epoch  6 | train 1.8128 | val 1.7405
epoch  7 | train 1.7370 | val 1.6862
epoch  8 | train 1.6770 | val 1.6328
epoch  9 | train 1.6218 | val 1.5892
epoch 10 | train 1.5768 | val 1.5533
epoch 11 | train 1.5350 | val 1.5276
epoch 12 | train 1.4990 | val 1.4999
epoch 13 | train 1.4731 | val 1.4805
epoch 14 | train 1.4424 | val 1.4611
epoch 15 | train 1.4128 | val 1.4524
epoch 16 | train 1.3893 | val 1.4424
epoch 17 | train 1.3707 | val 1.4222
epoch 18 | train 1.3526 | val 1.4163
epoch 19 | train 1.3316 | val 1.4073
epoch 20 | train 1.3154 | val 1.3993
epoch 21 | train 1.2979 | val 1.3938
epoch 22 | train 1.2880 | val 1.3827
epoch 23 | train 1.2712 | val 1.3734
epoch 24 | train 1.2550 | val 1.3744
epoch 25 | train 1.2443 | val 1.3805
epoch 26 | train 1.2330 | val 1.3707
e

## 5. Sampling

In [6]:
@torch.no_grad()
def sample_gpt(model, block_size, stoi, itos, device,
               start_text="Dorothy", max_new_tokens=1000,
               temperature=1.0, top_k=None):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        # Temperature scaling: lower is more deterministic (repetitive), higher is more varied
        logits = logits / temperature
        if top_k is not None:
            # Top-k sampling: keep only the k most likely tokens and mask the rest to -inf
            # so unlikely, odd tokens are never sampled
            values, _ = torch.topk(logits, top_k)
            logits[logits < values[:, [-1]]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

# temperature=0.8, top_k=40: a standard setting that is fairly deterministic but still varied
print(sample_gpt(model, block_size, stoi, itos, device,
                 start_text="Dorothy", max_new_tokens=1000,
                 temperature=0.8, top_k=40))

Dorothy crows pushed the blew came to the two ark Oz, so that the ground was while he had secret to her a beautiful
breasts back not came burning to this, for she knew go, what hurt
every discovered with straw, and and then he panted him up in their brilliancy.

“I’m terribly to all be a very when it. What she never seemed to do me to come back to
Kansas—but if he is goes, and when nothing a little girl, who was
was so just the Wicked Witch of the West, and then they tamed as
Dorothy could not be carried him to do this.

“If you wear the fear!” she replied.

“Oh, yes,” said the Tin Woodman. “You must tremble
away him, so she will protect care not hurt,” said the Lion.

“When I shall be have the rest until they come dome from the Wicked
Witch and the Tin Woodman.

“Where is a great many?” asked Dorothy.

“And I shall get my brains in,” answered the Tin Woodman. “But we can grant the
dish, dressed in places and head to wait out the old woman. At first the
decided her paint he felt in a g

## Summary

The model is trained on the first 80% of the Oz text and evaluated on the unseen last 20%, so the
validation loss measures generalization to held-out text. The loss curve and the sample above are
the results reported in the README.
